In [1]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video

from imaginaire.lazy_config import LazyCall as L
from cosmos_predict2.data.action_conditioned.uha_dataset import OxeUhaDataModule, NoEncoder
from imaginaire.lazy_config import instantiate
from cosmos_predict2.configs.expert.defaults.data_uha import oxe_uha_train_dataloader

2025-10-23 17:11:53.931316: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-23 17:11:53.985258: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-23 17:11:53.985300: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-23 17:11:53.986931: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-23 17:11:53.996461: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# transforms_conf = dict(
#     move_axis=True,
#     bytes_to_string=True,
#     adjust_type=None,
#     add_robot_information=False
# )
#
# language_encoders_conf = dict(
#     model_name=None
# )
#
# frame_transform_kwargs=dict(
#     image_augment_kwargs=dict(
#         primary=dict(
#             random_resized_crop=dict(
#                 scale=(0.8, 1.0),
#                 ratio=(0.9, 1.1)
#             ),
#             random_brightness=(0.1,),
#             random_contrast=(0.9, 1.1),
#             random_saturation=(0.9, 1.1),
#             random_hue=(0.05,),
#             augment_order=(
#                 "random_resized_crop",
#                 "random_brightness",
#                 "random_contrast",
#                 "random_saturation",
#                 "random_hue",
#             ),
#         ),
#         secondary=dict(
#             random_resized_crop=dict(
#                 scale=(0.8, 1.0),
#                 ratio=(0.9, 1.1)
#             ),
#             random_brightness=(0.1,),
#             random_contrast=(0.9, 1.1),
#             random_saturation=(0.9, 1.1),
#             random_hue=(0.05,),
#             augment_order=(
#                 "random_resized_crop",
#                 "random_brightness",
#                 "random_contrast",
#                 "random_saturation",
#                 "random_hue",
#             ),
#         ),
#         wrist=dict(
#             random_brightness=(0.1,),
#             random_contrast=(0.9, 1.1),
#             random_saturation=(0.9, 1.1),
#             random_hue=(0.05,),
#             augment_order=(
#                 "random_brightness",
#                 "random_contrast",
#                 "random_saturation",
#                 "random_hue",
#             ),
#         ),
#     ),
#     resize_size=dict(
#         primary=(176, 176),
#         secondary=(160, 160),  # not used
#         wrist=(84, 84),  # all black
#     ),
#     resize_size_future_obs=dict(
#         primary=(176, 176),
#         secondary=(160, 160),  # should be same as resize_size
#         wrist=(84, 84),
#     ),
#     num_parallel_calls=6,
# )
#
#
# DEBUG_DATASET = "bridge2"
# DEBUG_DATASET_MAPPING = {
#     "fractal": "fractal",
#     "bridge2": "bridge",
# }
#
# n_v_cond, n_v_out = 4 * 1 + 1, 4 * 5  # 4+1+20=25
# n_a_out = n_v_out
# n_latent_v_cond, n_latent_v_out = 1 * 1 + 1, 1 * 5  # 1+1+5=7
# horizon = n_v_cond + n_v_out # 25
# pad_before = n_v_cond - 1
# datasets_conf = dict(
#     DATA_NAME="bridge",  # ori: "fractal"
#     DATA_PATH="/home/geyuan/local_soft/huggingface/v1/",
#     load_camera_views=["primary"],  # ori: ["primary", "secondary", "wrist"],
#     load_proprio=True,  # ori: False
#     load_language_embeddings=True,  # ori: False
#     action_proprio_normalization_type="bounds",
#     interleaved_dataset_cfg=dict(
#         shuffle_buffer_size=5000,  # ori: 5000
#         balance_weights=True,
#         traj_transform_kwargs=dict(
#             goal_relabeling_strategy=None,
#             goal_relabeling_kwargs=dict(
#                 min_bound=20,
#                 max_bound=50,
#                 frame_diff=3
#             ),
#             window_size=n_v_cond,
#             action_horizon=n_a_out,
#             skip_unlabeled=True,
#             load_future_frames=True, # NOTE: ori: False
#         ),
#         frame_transform_kwargs=frame_transform_kwargs,
#         traj_transform_threads=16,
#         traj_read_threads=8,
#     )
# )
#
# uha_datamodule = OxeUhaDataModule(
#     transforms=transforms_conf,
#     language_encoders=language_encoders_conf,
#     datasets=datasets_conf,
#     batch_size=2,
#     drop_last=True,
#     # CosmosPredict2 specific
#     use_ori_uha_data_collate=False,
#     p_camera_drop=0.,
#     p_proprio_drop=0.,
#     state_t=n_latent_v_cond + n_latent_v_out,
# )
# uha_datamodule.prepare_data()
# uha_datamodule.setup()

train_dataloader = instantiate(
    oxe_uha_train_dataloader,
)

[DEBUG] OxeUhaDataModule prepare_data finished.


2025-10-23 17:11:57.979337: W external/local_tsl/tsl/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Couldn't resolve host name', error details: Could not resolve host: metadata.google.internal".


[DEBUG] Loading language embeddings from: /home/geyuan/datasets/OXEDROID/uha_process/lang_emb_t5xxl/fractal/t5_embeddings.npz
[DEBUG] Created static lookup with 599 embeddings, shape: (512, 1024)


2025-10-23 17:13:05.655369: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[WARNING  | tensorflow         ]: AutoGraph could not transform <function _gcd_import at 0x7f0b31fe7400> and will run it as-is.
Cause: Unable to locate the source code of <function _gcd_import at 0x7f0b31fe7400>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f0b31fe7400>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-10-23 17:13:07.962986: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# fractal20220817_data: ====================================================1.000000 #
######################################################################################

[DEBUG] Loading language embeddings from: /home/geyuan/datasets/OXEDROID/uha_process/lang_emb_t5xxl/fractal/t5_embeddings.npz
[DEBUG] Created static lookup with 599 embeddings, shape: (512, 1024)


2025-10-23 17:14:11.280295: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


[DEBUG] OxeUhaDataModule setup finished (main=True). Train len=189320, Info: {'train_dataset': {'fractal20220817_data': {'action': {'mean': array([ 0.00698757,  0.00626593, -0.01262512,  0.04333351, -0.0057562 ,
        0.00091303,  0.53542048]), 'std': array([0.06921133, 0.05970513, 0.07353117, 0.15610471, 0.13164456,
       0.14593813, 0.4971121 ]), 'max': array([ 2.99845934, 22.09052849,  2.75075245,  1.57063651,  1.53210866,
        1.56915224,  1.        ]), 'min': array([-2.02045202, -5.49789953, -2.03166342, -1.56991792, -1.56989217,
       -1.57041943,  0.        ]), 'p99': array([0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657,
       0.44796681, 1.        ]), 'p01': array([-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113,
       -0.43643461,  0.        ]), 'mask': array([ True,  True,  True,  True,  True,  True, False])}, 'num_transitions': array(3786400), 'num_trajectories': array(87212)}}, 'val_dataset': None}


In [7]:
from tqdm import tqdm

DEBUG_DATASET = "fractal"
vis_idx = 5
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch(DEBUG_DATASET, batch)
    if idx < vis_idx:
        continue
    print_batch(DEBUG_DATASET, batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                             | 0/189320 [00:00<?, ?it/s]/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/cosmos-predict2/lib/python3.10/site-packages/torch/utils/data/_utils/collate.py:285: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  return collate([torch.as_tensor(b) for b in batch], collate_fn_map=collate_fn_map)
  0%|                                                 | 1/189320 [00:10<537:50:25, 10.23s/it]

fractal: Dict, keys=['observation', 'task', 'action', 'action_pad_mask', 'future_frames']
--observation: Dict, keys=['image_primary', 'timestep', 'pad_mask_dict', 'timestep_pad_mask', 'task_completed']
----image_primary, <class 'torch.Tensor'>, shape=torch.Size([20, 5, 3, 144, 144]), min=0.0000, max=255.0000
----timestep, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=0.0000, max=69.0000
----pad_mask_dict: Dict, keys=['image_primary', 'timestep']
------image_primary, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=1.0000, max=1.0000
------timestep, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=1.0000, max=1.0000
----timestep_pad_mask, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=0.0000, max=1.0000
----task_completed, <class 'torch.Tensor'>, shape=torch.Size([20, 5, 20]), min=0.0000, max=1.0000
--task: Dict, keys=['language_instruction', 'language_key', 'language_embedding', 'pad_mask_dict']
----language_instruction: List, len=20, elem_type=<class 'str'

  0%|                                                  | 5/189320 [00:10<55:26:07,  1.05s/it]

fractal: Dict, keys=['observation', 'task', 'action', 'action_pad_mask', 'future_frames']
--observation: Dict, keys=['image_primary', 'timestep', 'pad_mask_dict', 'timestep_pad_mask', 'task_completed']
----image_primary, <class 'torch.Tensor'>, shape=torch.Size([20, 5, 3, 144, 144]), min=0.0000, max=255.0000
----timestep, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=0.0000, max=62.0000
----pad_mask_dict: Dict, keys=['image_primary', 'timestep']
------image_primary, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=1.0000, max=1.0000
------timestep, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=1.0000, max=1.0000
----timestep_pad_mask, <class 'torch.Tensor'>, shape=torch.Size([20, 5]), min=0.0000, max=1.0000
----task_completed, <class 'torch.Tensor'>, shape=torch.Size([20, 5, 20]), min=0.0000, max=1.0000
--task: Dict, keys=['language_instruction', 'language_key', 'language_embedding', 'pad_mask_dict']
----language_instruction: List, len=20, elem_type=<class 'str'

  0%|                                                 | 5/189320 [00:16<169:39:48,  3.23s/it]


In [17]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample

horizon = mv_sample['action'][0].shape[0]
save_image_or_video(
    mv_sample['video'][0, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=4
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]

def denorm_action(act):
    return (act + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)

save_action_as_image(
    denorm_action(mv_sample['action'])[0, :, ].cpu().numpy(),
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action0123456.mp4",
    fps=4,
)
# save_action_as_image(
#     denorm_action(mv_sample['action'])[:, 6:7],
#     f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
# )

save_action_as_image(
    mv_sample['agent_pos'][0, :, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )


Plotting action dynamic figures...
[DEBUG] save_3d_action_as_image: torch.Size([25, 3]) torch.float32 tensor(0.) tensor(0.)
[10-23 16:06:58|INFO|../../../../../../home/geyuan/code/cospred2nvidia/cosmos_predict2/utils/vis_helpers.py:194:save_3d_action_as_image] Saved 3D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_fractal_mv_agentpos012.png


In [8]:
""" Visualization Original Dataloader """
max_vis_len = 50

def save_view(in_sample_, batch_key_: str, view_key_: str):
    in_video_ = in_sample_[batch_key_][f'image_{view_key_}']  # (B,T,C,H,W)
    in_video_ = in_video_.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
    suffix = "_future" if "future" in batch_key_ else ""
    save_image_or_video(
        in_video_[:, :max_vis_len].to(torch.float32) / 255.,  # (c,t,h,w)
        f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_{view_key_}{suffix}.mp4",
        fps=10
    )

# 1. Views
save_view(in_sample, 'observation', 'primary')
# save_view(in_sample, 'observation', 'secondary')
# save_view(in_sample, 'observation', 'wrist')

save_view(in_sample, 'future_frames', 'primary')
# save_view(in_sample, 'future_frames', 'secondary')
# save_view(in_sample, 'future_frames', 'wrist')

# 2. Actions
in_action = in_sample['action'][:, -1].cpu().numpy()  # (B,T,H,D) -> (B,H,D), in [-1,1]
save_action_as_image(
    in_action[0, :max_vis_len, :],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_normed_action.mp4",
    fps=5
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]
in_action_unnormed = (in_action + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)
save_action_as_image(
    in_action_unnormed[0, :max_vis_len, :],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_action.mp4",
)

# 3. Language
in_language = in_sample['task']['language_instruction'][0]
in_lang_emb = in_sample['task']['language_embedding'][0]
print(in_language)
print(in_lang_emb[0, :10])

data = np.load(f"/home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/{DEBUG_DATASET}/t5_embeddings.npz", allow_pickle=True)
map_text_2_emb = data['text_to_embedding_map'].item()
print(map_text_2_emb[in_language][0, :10])

# # 4. Proprio
# in_proprio = in_sample['observation']['proprio']  # (B,T,D), in [-1,1]
# dataset_meta = uha_datamodule.dataset_info
# subdataset_meta = list(dataset_meta['train_dataset'].values())[0]
# meta_proprio_p01 = subdataset_meta['proprio']['p01']
# meta_proprio_p99 = subdataset_meta['proprio']['p99']
# in_proprio_unnormed = (in_proprio + 1) / 2 * (np.array(meta_proprio_p99) - np.array(meta_proprio_p01)) + np.array(meta_proprio_p01)
# save_action_as_image(
#     in_proprio_unnormed[0, :max_vis_len, :3],
#     f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_proprio.png",
# )


[DEBUG] save_action_as_image: (20, 7) float32 -0.66443133 1.0
Plotting action dynamic figures...
[DEBUG] save_action_as_image: (20, 7) float64 -0.2897669771211314 1.0
Plotting action dynamic figures...
move brown chip bag near blue plastic bottle
tensor([ 0.0501, -0.0445, -0.0726, -0.1345, -0.0499, -0.0963, -0.0511, -0.1353,
         0.1769, -0.1093])
[ 0.05008 -0.04446 -0.0726  -0.1345  -0.04987 -0.09625 -0.0511  -0.1353
  0.1769  -0.1093 ]
